In [ ]:
'''Курсовая работа на тему Анализ ходов фигуры Конь - Дракон(12) на шахматной доске'''

'''Импорт нужных библиотек:'''
import tkinter
import pygame

'''Создание окна для ввода N_L_K:'''

root = tkinter.Tk()
root.title('Курсовая работа "Шахматы"')
root.geometry('170x150')
root.configure(bg='chartreuse2')

'''Заполнение полей ввода:'''

N, N_entry = tkinter.Label(master=root, text='Размер доски (N):'), tkinter.Entry(width=15)
N.pack()
N_entry.pack()
L, L_entry = tkinter.Label(master=root, text='Фигур на расставление (L):'), tkinter.Entry(width=15)
L.pack()
L_entry.pack()
K, K_entry = tkinter.Label(master=root, text='Стоящие фигуры (K):'), tkinter.Entry(width=15)
K.pack()
K_entry.pack()

good_value, N, L, K = False, 0, 0, 0

'''Метод от которого будет выведено сообщение об ошибке:'''


def wrong_enter():
    wrong_root = tkinter.Tk()
    wrong_root.title('Ошибка!')
    wrong_root.geometry('240x50')
    tkinter.Label(master=wrong_root, text='Неверный формат ввода!').pack()
    tkinter.Button(master=wrong_root, text='Ок', command=lambda: wrong_root.destroy()).pack()
    wrong_root.mainloop()


'''Обработка N_L_K с проверкой на правильность введенных данных:'''


def enter_data():
    global good_value, N, L, K
    N, L, K = N_entry.get(), L_entry.get(), K_entry.get()
    root.destroy()
    if not N.isdigit() or not L.isdigit() or not K.isdigit():
        wrong_enter()
    else:
        N, L, K = int(N), int(L), int(K)
        good_value = True


enter_button = tkinter.Button(text="Ввести", command=enter_data)
enter_button.pack()
root.mainloop()

'''Проверка наличия ошибок через enter_data:'''

if not good_value:
    raise TypeError('Тип данных неверен')

'''Обработка введённых координат:'''


def enter_coords_for_k(entry_pole: list):
    for c in enter_line:
        line = c.get()
        coords_x_y = tuple(line.split())
        if len(coords_x_y) == 2 and False not in tuple(map(lambda x: x.isdigit(), coords_x_y)):
            chess_pices.append((int(coords_x_y[0]), int(coords_x_y[1])))
        else:
            wrong_enter()
    k_root.destroy()


chess_pices = []
k_root = tkinter.Tk()
k_root.geometry(f'150x{20 * (K + 1)}')
k_root.title('Координаты (K)')
k_root.configure(bg='DarkViolet')
enter_line = [tkinter.Entry(master=k_root) for j in range(K)]
for i in range(K):
    enter_line[i].pack()

'''Кнопка ввода c командой на активацию функции enter_coords_for_k:'''

enter_button = tkinter.Button(text="Ввод данных", command=lambda: enter_coords_for_k(enter_line))
enter_button.pack()
k_root.mainloop()

'''Цвета, используемые на доске:'''

RED = (200, 25, 25)
PURPLE = (240, 0, 255)
GREEN = (0, 255, 0)
LightSteelBlue4 = (110, 123, 139)
BLACK = {'black': (0, 0, 0)}
WHITE = {'white': (255, 255, 255)}

'''Подготовка к запуску шахматной доски:'''

chess = pygame.display.set_mode((500, 500))
pygame.display.set_caption('Шахматная доска')
chess.fill(LightSteelBlue4)
width = 2
cell_size = (500 - (N + 1) * width) / N
clock = pygame.time.Clock()


'''Класс клетки на доске:'''


class SquareCell(pygame.sprite.Sprite):
    def __init__(self, cel_size: float, color: str) -> None:
        super(SquareCell, self).__init__()
        self.surface = pygame.Surface((cel_size, cel_size))
        if color == 'white':
            self.surface.fill(WHITE['white'])
        else:
            self.surface.fill(BLACK['black'])
        self.rect = self.surface.get_rect()


'''Класс фигуры на доске:'''

class ChessFigure(pygame.sprite.Sprite):
    def __init__(self, x: int, y: int, type: str) -> None:
        self.x = x
        self.y = y
        self.type = type
        self.surface = pygame.Surface((cell_size, cell_size))
        if type == 'figure':
            self.surface.fill(RED)
            self.color = RED
        elif type == 'new figure':
            self.surface.fill(GREEN)
            self.color = GREEN
        else:
            self.surface.fill(PURPLE)
            self.color = PURPLE
        self.rect = self.surface.get_rect()
        self.pygame_x_coord = width * (x + 1) + cell_size * self.x
        self.pygame_y_coord = width * (y + 1) + cell_size * self.y

    def __repr__(self):
        if self.type == 'figure' or self.type == 'new figure':
            return 'Figure'
        else:
            return 'Cell unger battle '

'''Функция для поиска клетки под боем для фигуры Конь-Дракон (Король + Слон(на 3 клетки по диагонали)):'''


def cell_under_battle(size: int, x: int, y: int):
    beat = []

    '''Клетки рядом с фигурой:'''

    for row in range(-1, 2):
        for collumn in range(-1, 2):
            if 0 <= x + row < size and 0 <= y + collumn < size and (x + row, y + collumn) not in beat:
                beat.append((x + row, y + collumn))

    '''Клетки по диагонали:'''

    step = 2
    while step <= 3:
        for row in range(x - step, x + step + 1, step * 2):
            for col in range(y - step, y + step + 1, step * 2):
                if 0 <= row < size and 0 <= col < size and (row, col) not in beat:
                    beat.append((row, col))
        step += 1
    return beat
'''Визуализация шахматной доски:'''

process = True
while process:
    clock.tick(50)
    cell_line = []
    cell_coords_line = []
    for x in range(N):
        cell_line.append([])
        cell_coords_line.append([])
        for y in range(N):
            coord_x, coord_y = width * (x + 1) + cell_size * x, width * (y + 1) + cell_size * y
            if (x + y) % 2 == 0:
                cell_line[x].append(SquareCell(cell_size, 'white').surface)
            else:
                cell_line[x].append(SquareCell(cell_size, 'black').surface)
            cell_coords_line[x].append((coord_x, coord_y, coord_x + cell_size, coord_y + cell_size))
            chess.blit(source=cell_line[x][y], dest=(coord_x, coord_y))



    '''Расстановка К фигур:'''

    coords_b = []
    l_figures = []

    '''Цикл для отрисовки:'''

    for a in chess_pices:
        l_figures.append(a)
        i, j = a
        new_u_b_c = cell_under_battle(N, i, j)

        coords_b += list(filter(lambda x: x not in coords_b, new_u_b_c))
        '''Отрисовка клеток под боем:'''
        for q, w in new_u_b_c:
            cur_cell_u_b = ChessFigure(q, w, 'cell_under_battle')
            chess.blit(source=cur_cell_u_b.surface, dest=(cur_cell_u_b.pygame_x_coord, cur_cell_u_b.pygame_y_coord))

        '''Отрисовка фигур:'''

        chess_figure = ChessFigure(i, j, 'figure')
        chess.blit(source=chess_figure.surface, dest=(chess_figure.pygame_x_coord, chess_figure.pygame_y_coord))

    '''Реализация расчетов:'''

    all_coords = list((g, h) for g in range(N) for h in range(N))
    coords_not_under_battle = list(filter(lambda x: x not in coords_b, all_coords))

    '''Расчеты и расстановка L фигур:'''

    imm_turn = True
    sol_list = []

    '''Обратимся к методу бинарной строки для перебора вариантов:'''

    for variant in range(2 ** len(coords_not_under_battle)):
        binary = str(format(variant, 'b')).zfill(len(coords_not_under_battle))

        '''Если колличество фигур(1-ц) равно L, то эти строки проходят проверку:'''

        if binary.count('1') == L:
            t_b_coords = coords_b[::-1]
            validate = True
            sol_list.append([])

            '''Проверяем бинарную строку:'''

            for index in range(len(binary)):
                if binary[index] == '1':

                    '''Если клетка Не под боем, тогда:'''

                    pair_x_y = coords_not_under_battle[index]

                    '''Добавляем pair_x_y в решение и список уже занятых коорд-т для этого решения:'''

                    if pair_x_y not in t_b_coords:
                        sol_list[-1].append(pair_x_y)
                        t_b_coords += list(filter(lambda x: x not in t_b_coords, cell_under_battle(N, pair_x_y[0], pair_x_y[1])))
                    else:
                        '''Иначе, в этой бинарной строке решение не подходит'''
                        del sol_list[-1]
                        validate = False
                        break
            '''Если все нормально, добавляем решение в файл:'''
            if validate and imm_turn:
                '''Первое найденное решение выводится на экран доски:'''
                for sol_variant in sol_list:
                    for x_y in sol_variant:
                        for new_beaten_cell in cell_under_battle(N, x_y[0], x_y[1]):
                            cell = ChessFigure(new_beaten_cell[0], new_beaten_cell[1], 'cell_under_battle')
                            chess.blit(source=cell.surface, dest=(cell.pygame_x_coord, cell.pygame_y_coord))
                        chs_fig = ChessFigure(x_y[0], x_y[1], 'new figure')
                        chess.blit(source=chs_fig.surface, dest=(chs_fig.pygame_x_coord, chs_fig.pygame_y_coord))
                pygame.display.flip()
                imm_turn = False



    output_file = open('output.txt', 'w')
    close = False

    def output():
        global close
        figures = str(l_figures)[1:-1]
        for l in sol_list:
            output_file.write(figures + ', ' + str(l)[1:-1] + '\n')
        output_root.destroy()
        close = True

    output_root = tkinter.Tk()
    output_root.geometry('100x40')
    output_root.title("Вывод данных в файл")
    output_root.configure(bg='DarkViolet')
    output_button = tkinter.Button(master=output_root, text="Вывести данные", command=output)
    output_button.pack()

    if not close:
        output_root.mainloop()
    if close:
        break